# Comparación: LSTM vs XGBoost
**Universidad de los Andes - Semillero EMBS**

Este notebook compara los resultados de ambos enfoques.

## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

In [ ]:
plt.rcParams.update({
    'figure.facecolor':  "#ffffff",
    'axes.facecolor':    "#ffffff",
    'axes.edgecolor':    '#1a3a5c',
    'axes.labelcolor':   "#000000",
    'axes.titlecolor':   '#F0F2F5',
    'xtick.color':       "#000000",
    'ytick.color':       "#000000",
    'grid.color':        "#a4b0bc",
    'text.color':        "#000000",
    'font.family':       'monospace',
})

COLOR_LSTM = '#00a3e0'
COLOR_XGB = '#78be20'

## 2. Cargar resultados

In [ ]:
BASE = Path('..')
OUTPUTS = BASE / 'outputs'
METRICS = OUTPUTS / 'metrics'
FIGURES = OUTPUTS / 'figures'

# Cargar resultados de ambos modelos
df_lstm = pd.read_excel(METRICS / 'lstm_resultados_loso.xlsx')
df_xgb = pd.read_excel(METRICS / 'xgboost_resultados_loso.xlsx')

print('✓ Resultados cargados')
print(f'\nLSTM:')
print(df_lstm)
print(f'\nXGBoost:')
print(df_xgb)

## 3. Comparación estadística

In [ ]:
# Remover fila de PROMEDIO
df_lstm_data = df_lstm[df_lstm['Sujeto_test'] != 'PROMEDIO'].copy()
df_xgb_data = df_xgb[df_xgb['Sujeto_test'] != 'PROMEDIO'].copy()

sujetos = df_lstm_data['Sujeto_test'].tolist()

# Estadísticas
lstm_mean = df_lstm_data['Mejor_acc'].mean()
lstm_std = df_lstm_data['Mejor_acc'].std()
xgb_mean = df_xgb_data['Accuracy'].mean()
xgb_std = df_xgb_data['Accuracy'].std()

print('ESTADÍSTICAS COMPARATIVAS:')
print('='*70)
print(f'{"Modelo":<15} {"Mean Accuracy":<20} {"Std":<10} {"Min":<10} {"Max":<10}')
print('-'*70)
print(f'{"LSTM":<15} {lstm_mean:.4f}           {lstm_std:.4f}     {df_lstm_data["Mejor_acc"].min():.4f}     {df_lstm_data["Mejor_acc"].max():.4f}')
print(f'{"XGBoost":<15} {xgb_mean:.4f}           {xgb_std:.4f}     {df_xgb_data["Accuracy"].min():.4f}     {df_xgb_data["Accuracy"].max():.4f}')
print('='*70)

In [ ]:
# Comparación por sujeto
print('\nCOMPARACIÓN POR SUJETO:')
print('='*70)
print(f'{"Sujeto":<15} {"LSTM":<12} {"XGBoost":<12} {"Diferencia":<15} {"Mejor"}')
print('-'*70)

for i, sujeto in enumerate(sujetos):
    lstm_acc = df_lstm_data.iloc[i]['Mejor_acc']
    xgb_acc = df_xgb_data.iloc[i]['Accuracy']
    diff = xgb_acc - lstm_acc
    mejor = 'XGBoost' if diff > 0 else ('LSTM' if diff < 0 else 'Empate')
    signo = '+' if diff > 0 else ''
    
    print(f'{sujeto:<15} {lstm_acc:.4f}      {xgb_acc:.4f}      {signo}{diff:.4f}         {mejor}')

print('='*70)

## 4. Gráfico de comparación

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(sujetos))
width = 0.35

# Barras
bars1 = ax.bar(x - width/2, df_lstm_data['Mejor_acc'], width, 
               label='LSTM', color=COLOR_LSTM, alpha=0.9)
bars2 = ax.bar(x + width/2, df_xgb_data['Accuracy'], width, 
               label='XGBoost', color=COLOR_XGB, alpha=0.9)

# Configuración
ax.set_xlabel('Sujeto')
ax.set_ylabel('Accuracy')
ax.set_title('Comparación: LSTM vs XGBoost (LOSO)')
ax.set_xticks(x)
ax.set_xticklabels(sujetos, rotation=30, ha='right')
ax.axhline(0.5, color='#cc2244', linestyle='--', linewidth=1, label='Azar')
ax.legend()
ax.grid(axis='y')
ax.set_ylim(0, 1.05)

# Anotar valores
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / 'comparacion_lstm_xgboost.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0f14')
plt.show()

print('✓ Gráfico guardado: comparacion_lstm_xgboost.png')

## 5. Resumen ejecutivo

In [ ]:
print('RESUMEN EJECUTIVO')
print('='*70)

print(f'\n1. LSTM (Secuencias temporales):')
print(f'   - Accuracy promedio: {lstm_mean:.1%} ± {lstm_std:.1%}')
print(f'   - Mejor sujeto: {df_lstm_data.loc[df_lstm_data["Mejor_acc"].idxmax(), "Sujeto_test"]} '
      f'({df_lstm_data["Mejor_acc"].max():.1%})')
print(f'   - Peor sujeto: {df_lstm_data.loc[df_lstm_data["Mejor_acc"].idxmin(), "Sujeto_test"]} '
      f'({df_lstm_data["Mejor_acc"].min():.1%})')

print(f'\n2. XGBoost (Características TSFEL):')
print(f'   - Accuracy promedio: {xgb_mean:.1%} ± {xgb_std:.1%}')
print(f'   - Mejor sujeto: {df_xgb_data.loc[df_xgb_data["Accuracy"].idxmax(), "Sujeto_test"]} '
      f'({df_xgb_data["Accuracy"].max():.1%})')
print(f'   - Peor sujeto: {df_xgb_data.loc[df_xgb_data["Accuracy"].idxmin(), "Sujeto_test"]} '
      f'({df_xgb_data["Accuracy"].min():.1%})')

diff_mean = xgb_mean - lstm_mean
print(f'\n3. CONCLUSIÓN:')
if diff_mean > 0.05:
    print(f'   ✓ XGBoost supera a LSTM por {diff_mean:.1%}')
elif diff_mean < -0.05:
    print(f'   ✓ LSTM supera a XGBoost por {-diff_mean:.1%}')
else:
    print(f'   ≈ Desempeño similar (diferencia < 5%)')

victorias_xgb = (df_xgb_data['Accuracy'] > df_lstm_data['Mejor_acc']).sum()
victorias_lstm = (df_lstm_data['Mejor_acc'] > df_xgb_data['Accuracy']).sum()

print(f'\n   - XGBoost gana en {victorias_xgb}/5 sujetos')
print(f'   - LSTM gana en {victorias_lstm}/5 sujetos')

print('\n' + '='*70)

## 6. Feature Importance (Top características de XGBoost)

In [ ]:
try:
    df_importance = pd.read_excel(METRICS / 'xgboost_feature_importance.xlsx')
    
    print('\nTop 10 características más importantes:')
    print(df_importance.head(10).to_string(index=False))
    
    # Análisis por dominio
    def clasificar_dominio(feature_name):
        if any(x in feature_name.lower() for x in ['mean', 'std', 'variance', 'max', 'min', 'rms']):
            return 'Estadístico'
        elif any(x in feature_name.lower() for x in ['fft', 'frequency', 'spectral', 'power']):
            return 'Frecuencial'
        elif any(x in feature_name.lower() for x in ['zero', 'slope', 'diff']):
            return 'Temporal'
        else:
            return 'Otro'
    
    df_importance['dominio'] = df_importance['feature'].apply(clasificar_dominio)
    
    print('\n\nImportancia por dominio:')
    print('='*50)
    for dominio, imp in df_importance.groupby('dominio')['importance'].sum().sort_values(ascending=False).items():
        n_features = (df_importance['dominio'] == dominio).sum()
        print(f'{dominio:<20}: {imp:.4f} ({n_features} características)')
        
except FileNotFoundError:
    print('Warning: No se encontró xgboost_feature_importance.xlsx')

## ✓ Análisis completado

### Archivos generados:
- `outputs/figures/comparacion_lstm_xgboost.png`
- `outputs/figures/feature_importance_top15.png`
- `outputs/metrics/xgboost_resultados_loso.xlsx`
- `outputs/metrics/xgboost_feature_importance.xlsx`

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Genera matriz de confusión para LSTM
Universidad de los Andes - Semillero EMBS
"""

import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import butter, filtfilt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

# Configuración gráficos
plt.rcParams.update({
    'figure.facecolor':  "#ffffff",
    'axes.facecolor':    "#ffffff",
    'axes.edgecolor':    '#1a3a5c',
    'axes.labelcolor':   "#000000",
    'axes.titlecolor':   "#000000",
    'xtick.color':       "#000000",
    'ytick.color':       "#000000",
    'grid.color':        "#a4b0bc",
    'text.color':        "#000000",
    'font.family':       'monospace',
})

COLOR_INICIAL = '#00a3e0'
COLOR_FATIGA = '#78be20'

# Detectar dispositivo
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

# Parámetros
FPS = 29.97
VENTANA = 90
OVERLAP = 45
ANGULOS = ['Rodilla_I', 'Cadera_I', 'Tronco', 'Tobillo_I']
SUJETOS = ['P1', 'P2', 'P3', 'P4', 'P5']
CONDICIONES = ['Inicial', 'Fatiga']
BASE = Path('..')
OUTPUTS = BASE / 'outputs'
BATCH_SIZE = 16

print('='*70)
print('GENERANDO MATRIZ DE CONFUSIÓN - LSTM')
print('='*70)

# =============================================================================
# FUNCIONES (copiar del notebook original)
# =============================================================================

def extraer_punto(keypoints, indice):
    b = indice * 3
    return (keypoints[b], keypoints[b+1], keypoints[b+2])

def calcular_angulo(p1, p2, p3, umbral=0.1):
    if any(p[2] < umbral for p in [p1, p2, p3]):
        return None
    v1 = (p1[0]-p2[0], p1[1]-p2[1])
    v2 = (p3[0]-p2[0], p3[1]-p2[1])
    dot = v1[0]*v2[0] + v1[1]*v2[1]
    m1 = math.sqrt(v1[0]**2 + v1[1]**2)
    m2 = math.sqrt(v2[0]**2 + v2[1]**2)
    if m1 == 0 or m2 == 0:
        return None
    return round(math.degrees(math.acos(max(-1.0, min(1.0, dot/(m1*m2))))), 2)

def calcular_angulo_tronco(hombro, cadera, umbral=0.1):
    if hombro[2] < umbral or cadera[2] < umbral:
        return None
    vx = hombro[0] - cadera[0]
    vy = hombro[1] - cadera[1]
    mag = math.sqrt(vx**2 + vy**2)
    if mag == 0:
        return None
    dot = (vx*0 + vy*(-1)) / mag
    return round(math.degrees(math.acos(max(-1.0, min(1.0, dot)))), 2)

def seleccionar_ciclista(personas, ancho_frame=1920):
    if len(personas) == 1:
        return personas[0]['pose_keypoints_2d']
    centro_frame = ancho_frame / 2
    mejor_idx, menor_dist = 0, float('inf')
    for idx, persona in enumerate(personas):
        kp = persona['pose_keypoints_2d']
        puntos_x = [kp[i*3] for i in [1,2,5,8,9,12]
                    if kp[i*3+2] > 0.1 and kp[i*3] > 0]
        if not puntos_x:
            continue
        dist = abs(sum(puntos_x)/len(puntos_x) - centro_frame)
        if dist < menor_dist:
            menor_dist = dist
            mejor_idx = idx
    return personas[mejor_idx]['pose_keypoints_2d']

def butter_lowpass(serie, fc=6.0, fps=29.97, orden=4):
    nyquist = fps / 2.0
    b, a = butter(orden, fc/nyquist, btype='low', analog=False)
    serie_interp = serie.interpolate(method='linear', limit_direction='both')
    nan_mask = serie.isna()
    if serie_interp.notna().sum() < 15:
        return serie.values
    filtrada = filtfilt(b, a, serie_interp.values)
    filtrada[nan_mask] = np.nan
    return filtrada

def cargar_serie(sujeto, condicion, outputs_dir, fps=29.97):
    json_dir = outputs_dir / f'{sujeto}_{condicion}' / 'json'
    archivos = sorted([f for f in os.listdir(json_dir)
                       if f.endswith('_keypoints.json')])
    ancho_frame = 1920
    for nombre in archivos[:10]:
        try:
            with open(json_dir / nombre) as f:
                d = json.load(f)
            if d.get('people'):
                kp = d['people'][0]['pose_keypoints_2d']
                xs = [kp[i*3] for i in range(25) if kp[i*3] > 0]
                if xs:
                    ancho_frame = max(xs) * 1.1
                    break
        except Exception:
            continue
    filas = []
    for nombre in archivos:
        try:
            with open(json_dir / nombre) as f:
                datos = json.load(f)
        except Exception:
            continue
        if not datos.get('people'):
            filas.append({'Rodilla_I': None, 'Cadera_I': None,
                          'Tronco': None, 'Tobillo_I': None})
            continue
        kp = seleccionar_ciclista(datos['people'], ancho_frame)
        p = {n: extraer_punto(kp, n) for n in range(25)}
        filas.append({
            'Rodilla_I': calcular_angulo(p[12], p[13], p[14]),
            'Cadera_I': calcular_angulo(p[1], p[12], p[13]),
            'Tronco': calcular_angulo_tronco(p[5], p[12]),
            'Tobillo_I': calcular_angulo(p[13], p[14], p[19]),
        })
    df = pd.DataFrame(filas)
    for angulo in ANGULOS:
        df[angulo] = butter_lowpass(df[angulo], fps=fps)
    return df

def normalizar(df, MEDIA_GLOBAL, STD_GLOBAL):
    df_norm = df.copy()
    for angulo in ANGULOS:
        df_norm[angulo] = (df[angulo] - MEDIA_GLOBAL[angulo]) / STD_GLOBAL[angulo]
        df_norm[angulo] = df_norm[angulo].fillna(0.0)
    return df_norm

def generar_ventanas(df_norm, etiqueta, sujeto, ventana=90, overlap=45):
    valores = df_norm[ANGULOS].values
    paso = ventana - overlap
    muestras = []
    for inicio in range(0, len(valores) - ventana + 1, paso):
        fragmento = valores[inicio: inicio + ventana]
        if np.isnan(fragmento).mean() < 0.3:
            fragmento = np.nan_to_num(fragmento, nan=0.0)
            muestras.append({
                'X': fragmento.astype(np.float32),
                'y': etiqueta,
                'sujeto': sujeto,
            })
    return muestras

class CiclistaDataset(Dataset):
    def __init__(self, muestras):
        self.muestras = muestras
    def __len__(self):
        return len(self.muestras)
    def __getitem__(self, idx):
        m = self.muestras[idx]
        X = torch.tensor(m['X'], dtype=torch.float32)
        y = torch.tensor(m['y'], dtype=torch.float32)
        return X, y

class CapaAtencion(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.atencion = nn.Linear(hidden_size, 1, bias=False)
    def forward(self, salidas_lstm):
        scores = self.atencion(salidas_lstm)
        pesos = torch.softmax(scores, dim=1)
        contexto = (pesos * salidas_lstm).sum(dim=1)
        return contexto, pesos.squeeze(-1)

class LSTMAtencion(nn.Module):
    def __init__(self, n_angulos=4, hidden_size=32, n_capas=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_angulos,
            hidden_size=hidden_size,
            num_layers=n_capas,
            batch_first=True,
            dropout=dropout if n_capas > 1 else 0.0,
        )
        self.atencion = CapaAtencion(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.clasificador = nn.Linear(hidden_size, 1)
    def forward(self, x):
        salidas, _ = self.lstm(x)
        contexto, pesos = self.atencion(salidas)
        contexto = self.dropout(contexto)
        logit = self.clasificador(contexto).squeeze(-1)
        return logit, pesos

# =============================================================================
# CARGAR DATOS
# =============================================================================

print('\nCargando datos...')
datos_crudos = {}
for sujeto in SUJETOS:
    for condicion in CONDICIONES:
        key = f'{sujeto}_{condicion}'
        df = cargar_serie(sujeto, condicion, OUTPUTS, FPS)
        datos_crudos[key] = df

# Normalización
todos = pd.concat(list(datos_crudos.values()), ignore_index=True)
MEDIA_GLOBAL = {}
STD_GLOBAL = {}
for angulo in ANGULOS:
    MEDIA_GLOBAL[angulo] = todos[angulo].dropna().mean()
    STD_GLOBAL[angulo] = todos[angulo].dropna().std()

datos_norm = {key: normalizar(df, MEDIA_GLOBAL, STD_GLOBAL) 
              for key, df in datos_crudos.items()}

# Generar ventanas
todas_las_muestras = []
for sujeto in SUJETOS:
    for condicion, etiqueta in zip(CONDICIONES, [0, 1]):
        key = f'{sujeto}_{condicion}'
        muestras = generar_ventanas(datos_norm[key], etiqueta, sujeto, VENTANA, OVERLAP)
        todas_las_muestras.extend(muestras)

print(f'✓ {len(todas_las_muestras)} ventanas cargadas')

# =============================================================================
# CARGAR MODELOS Y GENERAR PREDICCIONES
# =============================================================================

print('\nCargando modelos y generando predicciones...')

y_true_total = []
y_pred_total = []

for sujeto_test in SUJETOS:
    # Cargar modelo
    modelo = LSTMAtencion(n_angulos=4, hidden_size=32, n_capas=2, dropout=0.3).to(DEVICE)
    ruta_modelo = OUTPUTS / 'models' / f'lstm_{sujeto_test}.pt'
    
    if not ruta_modelo.exists():
        print(f'⚠️  Modelo no encontrado: {ruta_modelo}')
        continue
    
    modelo.load_state_dict(torch.load(ruta_modelo, map_location=DEVICE))
    modelo.eval()
    
    # Obtener muestras de test
    test_muestras = [m for m in todas_las_muestras if m['sujeto'] == sujeto_test]
    test_loader = DataLoader(CiclistaDataset(test_muestras),
                              batch_size=BATCH_SIZE, shuffle=False)
    
    # Predicciones
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(DEVICE)
            logit, _ = modelo(X_batch)
            preds = (torch.sigmoid(logit) > 0.5).cpu().numpy().astype(int)
            y_true_total.extend(y_batch.numpy().astype(int))
            y_pred_total.extend(preds)
    
    print(f'  ✓ {sujeto_test}: {len(test_muestras)} muestras')

# =============================================================================
# GENERAR MATRIZ DE CONFUSIÓN
# =============================================================================

print('\n' + '='*70)
print('MATRIZ DE CONFUSIÓN AGREGADA - LSTM')
print('='*70)

# Calcular matriz
cm = confusion_matrix(y_true_total, y_pred_total)

print('\nMatriz de confusión:')
print(f'                Pred Inicial  Pred Fatiga')
print(f'Real Inicial         {cm[0,0]:3d}           {cm[0,1]:3d}')
print(f'Real Fatiga          {cm[1,0]:3d}           {cm[1,1]:3d}')

# Métricas
total = cm.sum()
accuracy = (cm[0,0] + cm[1,1]) / total
precision_inicial = cm[0,0] / (cm[0,0] + cm[1,0]) if (cm[0,0] + cm[1,0]) > 0 else 0
recall_inicial = cm[0,0] / (cm[0,0] + cm[0,1]) if (cm[0,0] + cm[0,1]) > 0 else 0
precision_fatiga = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0
recall_fatiga = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0

print(f'\nMétricas:')
print(f'  Accuracy total:      {accuracy:.3f}')
print(f'  Precisión Inicial:   {precision_inicial:.3f}')
print(f'  Recall Inicial:      {recall_inicial:.3f}')
print(f'  Precisión Fatiga:    {precision_fatiga:.3f}')
print(f'  Recall Fatiga:       {recall_fatiga:.3f}')

# =============================================================================
# VISUALIZACIÓN
# =============================================================================

fig, ax = plt.subplots(figsize=(8, 6))

# Normalizar a porcentajes
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

# Crear colormap personalizado morado
from matplotlib.colors import LinearSegmentedColormap
colors = ['#ffffff', '#e6d9f0', '#c9b3e0', '#ac8dd0', '#8f67c0', '#772583']
n_bins = 100
cmap_morado = LinearSegmentedColormap.from_list('morado', colors, N=n_bins)

# Heatmap
sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap=cmap_morado, 
            cbar_kws={'label': 'Percentage (%)'},
            xticklabels=['Normal', 'Fatigue'],
            yticklabels=['Normal', 'Fatigue'],
            ax=ax, linewidths=2, linecolor='#1a3a5c')

# Agregar conteos absolutos
for i in range(2):
    for j in range(2):
        text = ax.texts[i*2 + j]
        text.set_text(f'{cm_percent[i, j]:.1f}%\n({cm[i, j]})')
        text.set_fontsize(14)
        text.set_weight('bold')

ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_title('Confusion Matrix - LSTM with Attention (Aggregated LOSO)', fontsize=14, pad=20)

plt.tight_layout()
plt.savefig(OUTPUTS / 'figures' / 'lstm_confusion_matrix.png',
            dpi=150, bbox_inches='tight', facecolor='#ffffff')
plt.show()
print(f'\n✓ Figura guardada: outputs/figures/lstm_confusion_matrix.png')

# Reporte detallado
print('\n' + '='*70)
print('REPORTE DE CLASIFICACIÓN')
print('='*70)
print(classification_report(y_true_total, y_pred_total, 
                            target_names=['Inicial', 'Fatiga'],
                            digits=3))

print('='*70)
print('PROCESO COMPLETADO')
print('='*70)